<a href="https://colab.research.google.com/github/Jimmyqst/Path-Loss-ML-Optimization/blob/main/KNN_(Test).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **Test UPLINK**

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import linregress
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ==============================================================================
# 0. ESTILO DE TESIS (CONFIGURACIÓN MAESTRA)
# ==============================================================================
def configurar_estilo_tesis():
    sns.set_theme(style="whitegrid", context="paper")
    plt.rcParams.update({
        'font.family': 'serif',
        'axes.labelsize': 11,
        'axes.titlesize': 12,
        'xtick.labelsize': 10,
        'ytick.labelsize': 10,
        'legend.fontsize': 10,
        'figure.titlesize': 14,
        'grid.alpha': 0.3,
        'grid.linestyle': '--',
        'lines.linewidth': 2,
    })

# Color Oficial KNN (Morado)
COLOR_KNN = '#8e44ad'

configurar_estilo_tesis()

# ==============================================================================
# 1. CONFIGURACIÓN
# ==============================================================================
DIR_DATA   = '/content/drive/MyDrive/TESIS 100%/Datos_Entrenamiento_Finales/UpLink/'
DIR_MODELO = '/content/drive/MyDrive/TESIS 100%/KNN/Uplink/Entrenamiento/'
DIR_SALIDA = '/content/drive/MyDrive/TESIS 100%/KNN/Uplink/Prueba/Test_Final_Uniforme/'

if not os.path.exists(DIR_SALIDA): os.makedirs(DIR_SALIDA)

# Constantes Físicas
FREQ_MHZ = 915.0
H_GATEWAY_EMPIRICOS = 30.0
H_NODO_EMPIRICOS = 1.5
BIN_SIZE = 200

print("🚀 INICIANDO TEST GENERAL UPLINK - KNN (ESTILO TESIS CORREGIDO)...")

# ==============================================================================
# 2. CARGA DE DATOS Y MODELO
# ==============================================================================
try:
    ruta_test = os.path.join(DIR_DATA, 'Test_UpLink.csv')
    if not os.path.exists(ruta_test):
        ruta_test = os.path.join(DIR_DATA, 'Test_Uplink.csv')
    test_df = pd.read_csv(ruta_test)

    # Carga Modelo
    ruta_modelo = os.path.join(DIR_MODELO, 'Modelo_KNN_Uplink.pkl')
    if not os.path.exists(ruta_modelo):
        ruta_modelo = os.path.join(DIR_MODELO, 'Modelo_KNN_Fase1.pkl')
    if not os.path.exists(ruta_modelo):
         raise FileNotFoundError(f"No se encuentra el modelo en: {DIR_MODELO}")

    knn_model = joblib.load(ruta_modelo)
    print(f"✅ Modelo cargado: {os.path.basename(ruta_modelo)}")

except Exception as e:
    print(f"❌ Error Fatal Carga: {e}"); exit()

# ==============================================================================
# 3. ALINEACIÓN DINÁMICA DE FEATURES (ANTI-BUG SCIKIT-LEARN)
# ==============================================================================
# 1. Buscar SF disponible
posibles_sf = ['SpreedFactor_UpLink', 'SpreedFactor', 'SpreadingFactor', 'SF']
col_sf = next((c for c in posibles_sf if c in test_df.columns), None)

# 2. Buscar Power disponible
posibles_pwr = ['Power_UpLink', 'Power_Uplink', 'Power']
col_pwr = next((c for c in posibles_pwr if c in test_df.columns), None)

# 3. Buscar Target
posibles_target = ['Path_Loss_UpLink', 'Path_Loss_Uplink', 'Path_Loss']
TARGET = next((c for c in posibles_target if c in test_df.columns), None)
if TARGET and TARGET != 'Path_Loss_UpLink':
    test_df.rename(columns={TARGET: 'Path_Loss_UpLink'}, inplace=True)
    TARGET = 'Path_Loss_UpLink'

# 4. Alineación Estricta con la memoria del modelo
if hasattr(knn_model, 'feature_names_in_'):
    FEATURES = list(knn_model.feature_names_in_)
    print(f"   ℹ️ El modelo exige exactamente estas features: {FEATURES}")

    # Extraer cómo guardó el modelo las variables SF y Power
    sf_model = next((f for f in FEATURES if 'SpreedFactor' in f or 'SF' in f), None)
    pwr_model = next((f for f in FEATURES if 'Power' in f), None)

    if sf_model and col_sf and col_sf != sf_model:
        print(f"   🔄 Ajustando nombre: '{col_sf}' -> '{sf_model}'")
        test_df.rename(columns={col_sf: sf_model}, inplace=True)
        col_sf = sf_model

    if pwr_model and col_pwr and col_pwr != pwr_model:
        print(f"   🔄 Ajustando nombre: '{col_pwr}' -> '{pwr_model}'")
        test_df.rename(columns={col_pwr: pwr_model}, inplace=True)
        col_pwr = pwr_model
else:
    FEATURES = ['Distance', 'Gateway_Altitude', 'EndDevice_Altitude', col_sf, 'Terreno', col_pwr]

# Validación Final
missing = [c for c in FEATURES + [TARGET] if c not in test_df.columns]
if missing: raise ValueError(f"❌ Faltan columnas requeridas: {missing}")

# ==============================================================================
# 4. CÁLCULO DE PREDICCIONES
# ==============================================================================
print("📐 Calculando predicciones...")

# A. KNN
try:
    test_df['Pred_KNN'] = knn_model.predict(test_df[FEATURES])
except Exception as e:
    print(f"❌ Error predicción KNN: {e}"); exit()

# B. Empíricos
d_km = test_df['Distance'] / 1000.0; d_m = test_df['Distance']
h_b = H_GATEWAY_EMPIRICOS; h_m = H_NODO_EMPIRICOS; f = FREQ_MHZ

test_df['Pred_FSPL'] = 32.44 + 20*np.log10(f) + 20*np.log10(d_km)
g_f = 44.49*np.log10(f) - 4.78*(np.log10(f))**2
test_df['Pred_Ericsson'] = (36.2 + 30.2*np.log10(d_km) - 12.0*np.log10(h_b) + 0.1*np.log10(h_b)*np.log10(d_km) - 3.2*(np.log10(11.75*h_m))**2 + g_f)
lam = 300.0 / f
A_sui = 20 * np.log10((4 * np.pi * 100) / lam)
gamma_sui = 4.0 - 0.0065 * h_b + 17.1 / h_b
test_df['Pred_SUI'] = A_sui + 10 * gamma_sui * np.log10(d_m / 100.0) + 6.0 * np.log10(f / 2000.0) - 10.8 * np.log10(h_m / 2.0)
test_df['Pred_Egli'] = 88.0 + 20*np.log10(f) + 40*np.log10(d_km) - 20*np.log10(h_b) - 10*np.log10(h_m)

# FI y LNS
x_log = 10 * np.log10(d_m)
slope_fi, intercept_fi, _, _, _ = linregress(x_log, test_df[TARGET])
test_df['Pred_FI'] = intercept_fi + slope_fi * x_log

fspl_d0 = 32.44 + 20*np.log10(f)
y_diff = test_df[TARGET] - fspl_d0
x_term = 10 * np.log10(d_km)
n_opt = np.sum(x_term * y_diff) / np.sum(x_term**2)
test_df['Pred_LNS'] = fspl_d0 + n_opt * x_term

# ==============================================================================
# 5. MÉTRICAS GLOBALES
# ==============================================================================
def calc_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {'RMSE': np.sqrt(mse), 'MSE': mse, 'MAE': mean_absolute_error(y_true, y_pred), 'R2': r2_score(y_true, y_pred)}

nombres = ['KNN', 'Floating Intercept', 'Log-Normal Shad.', 'SUI', 'Ericsson', 'Egli', 'FSPL']
cols_res = ['Pred_KNN', 'Pred_FI', 'Pred_LNS', 'Pred_SUI', 'Pred_Ericsson', 'Pred_Egli', 'Pred_FSPL']

metrics_glob = {}
for n, c in zip(nombres, cols_res):
    metrics_glob[n] = calc_metrics(test_df[TARGET], test_df[c])

# ==============================================================================
# 6. TABLAS Y GRÁFICAS GLOBALES
# ==============================================================================
print("📊 Generando Reportes Globales...")

def save_table(data, cols, title, fname):
    plt.figure(figsize=(8, len(data)*0.5 + 1.2))
    ax = plt.gca(); ax.axis('off')
    t = ax.table(cellText=data, colLabels=cols, loc='center', cellLoc='center')
    t.auto_set_font_size(False); t.set_fontsize(11); t.scale(1, 1.8)
    for (i, j), cell in t.get_celld().items():
        if i == 0:
            cell.set_facecolor(COLOR_KNN) # Morado KNN
            cell.set_text_props(color='white', weight='bold')
        else:
            cell.set_facecolor('#f8f9fa')
            cell.set_edgecolor('#dddddd')
    plt.title(title, fontsize=14, fontweight='bold', y=0.98)
    plt.savefig(os.path.join(DIR_SALIDA, fname), dpi=300, bbox_inches='tight'); plt.close()

# 1. Tabla KNN Global
knn_metrics = [[k, f"{v:.4f}"] for k, v in metrics_glob['KNN'].items()]
save_table(knn_metrics, ['Métrica', 'Valor'], 'Desempeño Global KNN (Uplink)', '1_Tabla_Global_KNN.png')

# 1B. Barras Solo KNN
plt.figure(figsize=(8, 6))
m_knn = metrics_glob['KNN']
bars = plt.bar(list(m_knn.keys()), list(m_knn.values()), color=COLOR_KNN, edgecolor='black', width=0.6)
plt.bar_label(bars, fmt='%.2f', padding=3, fontweight='bold')
plt.title('Métricas Globales KNN (Uplink)', fontweight='bold')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.savefig(os.path.join(DIR_SALIDA, '1B_Barras_SoloKNN_Global.png'), dpi=300); plt.close()

# 2. Tabla Comparativa
comp_data = []
for n in nombres:
    v = metrics_glob[n]
    comp_data.append([n, f"{v['RMSE']:.2f}", f"{v['MSE']:.2f}", f"{v['MAE']:.2f}", f"{v['R2']:.3f}"])
save_table(comp_data, ['Modelo', 'RMSE', 'MSE', 'MAE', 'R2'], 'Comparativa Global de Modelos', '2_Tabla_Global_Comparativa.png')

# 3. Barras Comparativas
df_m = pd.DataFrame(metrics_glob).T
for met in ['RMSE', 'MSE', 'MAE', 'R2']:
    plt.figure(figsize=(10, 6))
    df_s = df_m.sort_values(met, ascending=(met!='R2'))
    colors = [COLOR_KNN if x == 'KNN' else '#95a5a6' for x in df_s.index]
    bars = plt.bar(df_s.index, df_s[met], color=colors, edgecolor='black', alpha=0.9)
    plt.bar_label(bars, fmt='%.2f', fontsize=10, fontweight='bold')
    plt.title(f'Comparativa de Modelos - {met} (Uplink)', fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(os.path.join(DIR_SALIDA, f'3_Barras_Comp_{met}.png'), dpi=300); plt.close()

# 4. Curvas Comparativas
plt.figure(figsize=(12, 7))
plt.scatter(test_df['Distance'], test_df[TARGET], c='gray', alpha=0.2, s=15, label='Medición Real')

X_s = np.linspace(test_df['Distance'].min(), test_df['Distance'].max(), 300)
d_km_s = X_s/1000.0

plt.plot(X_s, 32.44+20*np.log10(f)+20*np.log10(d_km_s), 'k:', lw=1.5, label='FSPL')
plt.plot(X_s, intercept_fi+slope_fi*(10*np.log10(X_s)), color='#e67e22', lw=2, label='FI')
plt.plot(X_s, fspl_d0+n_opt*(10*np.log10(d_km_s)), color='#2ecc71', lw=2, label='LNS')

# Tendencia KNN
test_df['Bin'] = (np.round(test_df['Distance']/100)*100).astype(int)
knn_trend = test_df.groupby('Bin')['Pred_KNN'].mean().sort_index()
plt.plot(knn_trend.index, knn_trend.values, color=COLOR_KNN, lw=3.5, label='KNN (Tendencia)')

plt.title('Comparativa de Propagación Global (Uplink)', fontweight='bold')
plt.xlabel('Distancia (m)', fontweight='bold'); plt.ylabel('Path Loss (dB)', fontweight='bold')
plt.legend(loc='lower right') # <-- LEYENDA ASEGURADA
plt.grid(True, linestyle='--', alpha=0.5)
plt.savefig(os.path.join(DIR_SALIDA, '4_Curvas_Comparativas.png'), dpi=300); plt.close()

# 5. HISTOGRAMA DE ERRORES (GLOBAL)
residuos_global = test_df[TARGET] - test_df['Pred_KNN']

plt.figure(figsize=(10, 6))
sns.histplot(residuos_global, kde=True, color=COLOR_KNN, bins=40, line_kws={'linewidth': 2}, alpha=0.6)
plt.axvline(0, color='black', linestyle='--', linewidth=1.5, label='Ideal (0 dB)')
plt.axvline(residuos_global.mean(), color='red', linestyle=':', linewidth=1.5,
            label=f'Media: {residuos_global.mean():.2f} dB')

plt.title(f'Distribución de Errores Globales - KNN\n(Std Dev: {residuos_global.std():.2f} dB)',
          fontsize=14, fontweight='bold')
plt.xlabel('Error (Real - Predicho) [dB]', fontweight='bold')
plt.ylabel('Frecuencia', fontweight='bold')
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.savefig(os.path.join(DIR_SALIDA, '5_Histograma_Errores_Global.png'), dpi=300, bbox_inches='tight')
plt.close()

# 6. SCATTER GLOBAL (COMPLETANDO EL PAQUETE)
plt.figure(figsize=(10, 6))
plt.scatter(test_df['Distance'], test_df[TARGET], c='gray', alpha=0.5, label='Real')
plt.scatter(test_df['Distance'], test_df['Pred_KNN'], c=COLOR_KNN, marker='x', alpha=0.9, label='KNN')
xr = np.linspace(test_df['Distance'].min(), test_df['Distance'].max(), 100)
plt.plot(xr, 32.44 + 20*np.log10(f) + 20*np.log10(xr/1000), 'k--', label='FSPL')
plt.title('KNN vs Real (Global Uplink)', fontweight='bold')
plt.legend(loc='lower right') # <-- LEYENDA ASEGURADA
plt.grid(True, linestyle='--', alpha=0.5)
plt.savefig(os.path.join(DIR_SALIDA, '6_Scatter_Global.png'), dpi=300); plt.close()

# ==============================================================================
# 7. ANÁLISIS POR CONFIGURACIÓN (LOOP)
# ==============================================================================
print("📂 Generando Análisis por Configuración...")
DIR_CFG = os.path.join(DIR_SALIDA, 'Por_Configuracion')
if not os.path.exists(DIR_CFG): os.makedirs(DIR_CFG)

config_summary = []

for (sf, pwr), grp in test_df.groupby([col_sf, col_pwr]):
    if len(grp) < 10: continue

    nm = f"SF{sf}_P{pwr}"
    m_loc = {}
    for n, c in zip(nombres, cols_res):
        m_loc[n] = calc_metrics(grp[TARGET], grp[c])

    config_summary.append([nm, m_loc['KNN']['RMSE'], m_loc['KNN']['MSE'], m_loc['KNN']['R2'], len(grp)])

    # 1. Tabla KNN Local
    d_knn_loc = [[k, f"{v:.4f}"] for k, v in m_loc['KNN'].items()]
    save_table(d_knn_loc, ['Métrica', 'Valor'], f'Métricas KNN - {nm}', os.path.join(DIR_CFG, f'Tabla_SoloKNN_{nm}.png'))

    # 2. Barras KNN Local
    plt.figure(figsize=(6, 4))
    vals = list(m_loc['KNN'].values())
    keys = list(m_loc['KNN'].keys())
    bars = plt.bar(keys, vals, color=COLOR_KNN, edgecolor='black')
    plt.bar_label(bars, fmt='%.2f', fontweight='bold')
    plt.title(f'Desempeño KNN Uplink - {nm}', fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(DIR_CFG, f'Barras_SoloKNN_{nm}.png'), dpi=300); plt.close()

    # 3. Barras Comparativas Local (RMSE)
    df_loc = pd.DataFrame(m_loc).T
    plt.figure(figsize=(10, 5))
    df_s = df_loc.sort_values('RMSE')
    colors = [COLOR_KNN if x == 'KNN' else '#95a5a6' for x in df_s.index]
    bars = plt.bar(df_s.index, df_s['RMSE'], color=colors, edgecolor='black')
    plt.bar_label(bars, fmt='%.2f', fontweight='bold')
    plt.title(f'Comparativa RMSE - {nm}', fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(os.path.join(DIR_CFG, f'Barras_Comp_RMSE_{nm}.png'), dpi=300); plt.close()

    # 4. Scatter Local
    plt.figure(figsize=(10, 6))
    plt.scatter(grp['Distance'], grp[TARGET], c='gray', alpha=0.5, label='Real')
    plt.scatter(grp['Distance'], grp['Pred_KNN'], c=COLOR_KNN, marker='x', alpha=0.9, label='Pred. KNN')

    # Línea FSPL referencia
    xr = np.linspace(grp['Distance'].min(), grp['Distance'].max(), 100)
    plt.plot(xr, 32.44+20*np.log10(f)+20*np.log10(xr/1000), 'k--', alpha=0.5, label='FSPL')

    plt.title(f'Predicción vs Real Uplink - {nm}', fontweight='bold')
    plt.xlabel('Distancia (m)'); plt.ylabel('Path Loss (dB)')
    plt.legend(loc='lower right') # <-- LEYENDA ASEGURADA
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.savefig(os.path.join(DIR_CFG, f"Scatter_{nm}.png"), dpi=300, bbox_inches='tight'); plt.close()

    # 5. Boxplot Local
    plt.figure(figsize=(12, 6))
    bins = range(int(grp['Distance'].min()), int(grp['Distance'].max()) + BIN_SIZE + 1, BIN_SIZE)
    grp_copy = grp.copy()
    grp_copy['Bin'] = pd.cut(grp_copy['Distance'], bins=bins)

    if not grp_copy['Bin'].isnull().all():
        sns.boxplot(x='Bin', y='Pred_KNN', data=grp_copy, color='#af7ac5') # Morado clarito
        plt.title(f'Distribución KNN por Distancia - {nm}', fontweight='bold')
        plt.xticks(rotation=45)
        plt.grid(axis='y', linestyle='--', alpha=0.5)
        plt.tight_layout()
        plt.savefig(os.path.join(DIR_CFG, f"Boxplot_{nm}.png"), dpi=300); plt.close()

# Guardar resumen CSV
pd.DataFrame(config_summary, columns=['Config', 'RMSE', 'MSE', 'R2', 'Muestras']).to_csv(os.path.join(DIR_SALIDA, 'Resumen_Configs.csv'), index=False)

print(f"\n✅ TEST KNN UPLINK GENERAL COMPLETADO.")
print(f"   📂 Resultados en: {DIR_SALIDA}")

# **DOWNLINK**

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import linregress
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ==============================================================================
# 0. ESTILO DE TESIS (CONFIGURACIÓN MAESTRA)
# ==============================================================================
def configurar_estilo_tesis():
    sns.set_theme(style="whitegrid", context="paper")
    plt.rcParams.update({
        'font.family': 'serif',
        'axes.labelsize': 11,
        'axes.titlesize': 12,
        'xtick.labelsize': 10,
        'ytick.labelsize': 10,
        'legend.fontsize': 10,
        'figure.titlesize': 14,
        'grid.alpha': 0.3,
        'grid.linestyle': '--',
        'lines.linewidth': 2,
    })

# Color Oficial KNN (Morado)
COLOR_KNN = '#8e44ad'

configurar_estilo_tesis()

# ==============================================================================
# 1. CONFIGURACIÓN (RUTAS DOWNLINK)
# ==============================================================================
DIR_DATA   = '/content/drive/MyDrive/TESIS 100%/Datos_Entrenamiento_Finales/DownLink/'
DIR_MODELO = '/content/drive/MyDrive/TESIS 100%/KNN/Downlink/Entrenamiento/'
DIR_SALIDA = '/content/drive/MyDrive/TESIS 100%/KNN/Downlink/Prueba/Test_Final_Uniforme/'

if not os.path.exists(DIR_SALIDA): os.makedirs(DIR_SALIDA)

# Constantes Físicas
FREQ_MHZ = 915.0
H_GATEWAY_EMPIRICOS = 30.0
H_NODO_EMPIRICOS = 1.5
BIN_SIZE = 200

print("🚀 INICIANDO TEST GENERAL DOWNLINK - KNN (ESTILO TESIS CORREGIDO)...")

# ==============================================================================
# 2. CARGA DE DATOS Y MODELO
# ==============================================================================
try:
    ruta_test = os.path.join(DIR_DATA, 'Test_DownLink.csv')
    if not os.path.exists(ruta_test):
        ruta_test = os.path.join(DIR_DATA, 'Test_Downlink.csv')
    test_df = pd.read_csv(ruta_test)

    # Carga Modelo
    ruta_modelo = os.path.join(DIR_MODELO, 'Modelo_KNN_Downlink.pkl')
    if not os.path.exists(ruta_modelo):
        ruta_modelo = os.path.join(DIR_MODELO, 'Modelo_KNN_Fase1.pkl')
    if not os.path.exists(ruta_modelo):
        posibles = [f for f in os.listdir(DIR_MODELO) if f.endswith('.pkl')]
        if posibles: ruta_modelo = os.path.join(DIR_MODELO, posibles[0])
        else: raise FileNotFoundError(f"No se encuentra el modelo en: {DIR_MODELO}")

    knn_model = joblib.load(ruta_modelo)
    print(f"✅ Modelo cargado: {os.path.basename(ruta_modelo)}")

except Exception as e:
    print(f"❌ Error Fatal Carga: {e}"); exit()

# ==============================================================================
# 3. ALINEACIÓN DINÁMICA DE FEATURES (ANTI-BUG SCIKIT-LEARN)
# ==============================================================================
# 1. Buscar SF disponible (Atrapando arrastres de Uplink)
posibles_sf = ['SpreedFactor_UpLink', 'SpreddFactor_Uplink', 'SpreedFactor_DownLink', 'SpreedFactor_Downlink', 'SpreedFactor', 'SpreadingFactor', 'SF']
col_sf = next((c for c in posibles_sf if c in test_df.columns), None)

# 2. Buscar Power disponible
posibles_pwr = ['Power_DownLink', 'Power_Downlink', 'Power_UpLink', 'Power']
col_pwr = next((c for c in posibles_pwr if c in test_df.columns), None)

# 3. Buscar Target
posibles_target = ['Path_Loss_DownLink', 'PathLoss_DownLink', 'Path_Loss_Downlink', 'Path_Loss_UpLink', 'Path_Loss']
TARGET = next((c for c in posibles_target if c in test_df.columns), None)
if TARGET and TARGET != 'Path_Loss_DownLink':
    test_df.rename(columns={TARGET: 'Path_Loss_DownLink'}, inplace=True)
    TARGET = 'Path_Loss_DownLink'

# 4. Alineación Estricta con la memoria del modelo
if hasattr(knn_model, 'feature_names_in_'):
    FEATURES = list(knn_model.feature_names_in_)
    print(f"   ℹ️ El modelo exige exactamente estas features: {FEATURES}")

    # Extraer cómo guardó el modelo las variables SF y Power
    sf_model = next((f for f in FEATURES if 'SpreedFactor' in f or 'SF' in f), None)
    pwr_model = next((f for f in FEATURES if 'Power' in f), None)

    if sf_model and col_sf and col_sf != sf_model:
        print(f"   🔄 Ajustando nombre: '{col_sf}' -> '{sf_model}'")
        test_df.rename(columns={col_sf: sf_model}, inplace=True)
        col_sf = sf_model

    if pwr_model and col_pwr and col_pwr != pwr_model:
        print(f"   🔄 Ajustando nombre: '{col_pwr}' -> '{pwr_model}'")
        test_df.rename(columns={col_pwr: pwr_model}, inplace=True)
        col_pwr = pwr_model
else:
    FEATURES = ['Distance', 'Gateway_Altitude', 'EndDevice_Altitude', col_sf, 'Terreno', col_pwr]

# Validación Final
missing = [c for c in FEATURES + [TARGET] if c not in test_df.columns]
if missing: raise ValueError(f"❌ Faltan columnas requeridas: {missing}")

# ==============================================================================
# 4. CÁLCULO DE PREDICCIONES
# ==============================================================================
print("📐 Calculando predicciones Downlink...")

# A. KNN
try:
    test_df['Pred_KNN'] = knn_model.predict(test_df[FEATURES])
except Exception as e:
    print(f"❌ Error predicción KNN: {e}"); exit()

# B. Empíricos
d_km = test_df['Distance'] / 1000.0; d_m = test_df['Distance']
h_b = H_GATEWAY_EMPIRICOS; h_m = H_NODO_EMPIRICOS; f = FREQ_MHZ

test_df['Pred_FSPL'] = 32.44 + 20*np.log10(f) + 20*np.log10(d_km)
g_f = 44.49*np.log10(f) - 4.78*(np.log10(f))**2
test_df['Pred_Ericsson'] = (36.2 + 30.2*np.log10(d_km) - 12.0*np.log10(h_b) + 0.1*np.log10(h_b)*np.log10(d_km) - 3.2*(np.log10(11.75*h_m))**2 + g_f)
lam = 300.0 / f
A_sui = 20 * np.log10((4 * np.pi * 100) / lam)
gamma_sui = 4.0 - 0.0065 * h_b + 17.1 / h_b
test_df['Pred_SUI'] = A_sui + 10 * gamma_sui * np.log10(d_m / 100.0) + 6.0 * np.log10(f / 2000.0) - 10.8 * np.log10(h_m / 2.0)
test_df['Pred_Egli'] = 88.0 + 20*np.log10(f) + 40*np.log10(d_km) - 20*np.log10(h_b) - 10*np.log10(h_m)

# FI y LNS
x_log = 10 * np.log10(d_m)
slope_fi, intercept_fi, _, _, _ = linregress(x_log, test_df[TARGET])
test_df['Pred_FI'] = intercept_fi + slope_fi * x_log

fspl_d0 = 32.44 + 20*np.log10(f)
y_diff = test_df[TARGET] - fspl_d0
x_term = 10 * np.log10(d_km)
n_opt = np.sum(x_term * y_diff) / np.sum(x_term**2)
test_df['Pred_LNS'] = fspl_d0 + n_opt * x_term

# ==============================================================================
# 5. MÉTRICAS GLOBALES
# ==============================================================================
def calc_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {'RMSE': np.sqrt(mse), 'MSE': mse, 'MAE': mean_absolute_error(y_true, y_pred), 'R2': r2_score(y_true, y_pred)}

nombres = ['KNN', 'Floating Intercept', 'Log-Normal Shad.', 'SUI', 'Ericsson', 'Egli', 'FSPL']
cols_res = ['Pred_KNN', 'Pred_FI', 'Pred_LNS', 'Pred_SUI', 'Pred_Ericsson', 'Pred_Egli', 'Pred_FSPL']

metrics_glob = {}
for n, c in zip(nombres, cols_res):
    metrics_glob[n] = calc_metrics(test_df[TARGET], test_df[c])

# ==============================================================================
# 6. TABLAS Y GRÁFICAS GLOBALES
# ==============================================================================
print("📊 Generando Reportes Globales...")

def save_table(data, cols, title, fname):
    plt.figure(figsize=(8, len(data)*0.5 + 1.2))
    ax = plt.gca(); ax.axis('off')
    t = ax.table(cellText=data, colLabels=cols, loc='center', cellLoc='center')
    t.auto_set_font_size(False); t.set_fontsize(11); t.scale(1, 1.8)
    for (i, j), cell in t.get_celld().items():
        if i == 0:
            cell.set_facecolor(COLOR_KNN) # Morado KNN
            cell.set_text_props(color='white', weight='bold')
        else:
            cell.set_facecolor('#f8f9fa')
            cell.set_edgecolor('#dddddd')
    plt.title(title, fontsize=14, fontweight='bold', y=0.98)
    plt.savefig(os.path.join(DIR_SALIDA, fname), dpi=300, bbox_inches='tight'); plt.close()

# 1. Tabla KNN Global
knn_metrics = [[k, f"{v:.4f}"] for k, v in metrics_glob['KNN'].items()]
save_table(knn_metrics, ['Métrica', 'Valor'], 'Desempeño Global KNN (Downlink)', '1_Tabla_Global_KNN.png')

# 1B. Barras Solo KNN
plt.figure(figsize=(8, 6))
m_knn = metrics_glob['KNN']
bars = plt.bar(list(m_knn.keys()), list(m_knn.values()), color=COLOR_KNN, edgecolor='black', width=0.6)
plt.bar_label(bars, fmt='%.2f', padding=3, fontweight='bold')
plt.title('Métricas Globales KNN (Downlink)', fontweight='bold')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.savefig(os.path.join(DIR_SALIDA, '1B_Barras_SoloKNN_Global.png'), dpi=300); plt.close()

# 2. Tabla Comparativa
comp_data = []
for n in nombres:
    v = metrics_glob[n]
    comp_data.append([n, f"{v['RMSE']:.2f}", f"{v['MSE']:.2f}", f"{v['MAE']:.2f}", f"{v['R2']:.3f}"])
save_table(comp_data, ['Modelo', 'RMSE', 'MSE', 'MAE', 'R2'], 'Comparativa Global de Modelos', '2_Tabla_Global_Comparativa.png')

# 3. Barras Comparativas
df_m = pd.DataFrame(metrics_glob).T
for met in ['RMSE', 'MSE', 'MAE', 'R2']:
    plt.figure(figsize=(10, 6))
    df_s = df_m.sort_values(met, ascending=(met!='R2'))
    colors = [COLOR_KNN if x == 'KNN' else '#95a5a6' for x in df_s.index]
    bars = plt.bar(df_s.index, df_s[met], color=colors, edgecolor='black', alpha=0.9)
    plt.bar_label(bars, fmt='%.2f', fontsize=10, fontweight='bold')
    plt.title(f'Comparativa de Modelos - {met} (Downlink)', fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(os.path.join(DIR_SALIDA, f'3_Barras_Comp_{met}.png'), dpi=300); plt.close()

# 4. Curvas Comparativas
plt.figure(figsize=(12, 7))
plt.scatter(test_df['Distance'], test_df[TARGET], c='gray', alpha=0.2, s=15, label='Medición Real')

X_s = np.linspace(test_df['Distance'].min(), test_df['Distance'].max(), 300)
d_km_s = X_s/1000.0

plt.plot(X_s, 32.44+20*np.log10(f)+20*np.log10(d_km_s), 'k:', lw=1.5, label='FSPL')
plt.plot(X_s, intercept_fi+slope_fi*(10*np.log10(X_s)), color='#e67e22', lw=2, label='FI')
plt.plot(X_s, fspl_d0+n_opt*(10*np.log10(d_km_s)), color='#2ecc71', lw=2, label='LNS')

# Tendencia KNN
test_df['Bin'] = (np.round(test_df['Distance']/100)*100).astype(int)
knn_trend = test_df.groupby('Bin')['Pred_KNN'].mean().sort_index()
plt.plot(knn_trend.index, knn_trend.values, color=COLOR_KNN, lw=3.5, label='KNN (Tendencia)')

plt.title('Comparativa de Propagación Global (Downlink)', fontweight='bold')
plt.xlabel('Distancia (m)', fontweight='bold'); plt.ylabel('Path Loss (dB)', fontweight='bold')
plt.legend(loc='lower right') # <-- LEYENDA ASEGURADA
plt.grid(True, linestyle='--', alpha=0.5)
plt.savefig(os.path.join(DIR_SALIDA, '4_Curvas_Comparativas.png'), dpi=300); plt.close()

# 5. HISTOGRAMA DE ERRORES (GLOBAL)
residuos_global = test_df[TARGET] - test_df['Pred_KNN']

plt.figure(figsize=(10, 6))
sns.histplot(residuos_global, kde=True, color=COLOR_KNN, bins=40, line_kws={'linewidth': 2}, alpha=0.6)
plt.axvline(0, color='black', linestyle='--', linewidth=1.5, label='Ideal (0 dB)')
plt.axvline(residuos_global.mean(), color='red', linestyle=':', linewidth=1.5,
            label=f'Media: {residuos_global.mean():.2f} dB')

plt.title(f'Distribución de Errores Globales - KNN\n(Std Dev: {residuos_global.std():.2f} dB)',
          fontsize=14, fontweight='bold')
plt.xlabel('Error (Real - Predicho) [dB]', fontweight='bold')
plt.ylabel('Frecuencia', fontweight='bold')
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.savefig(os.path.join(DIR_SALIDA, '5_Histograma_Errores_Global.png'), dpi=300, bbox_inches='tight')
plt.close()

# 6. SCATTER GLOBAL (COMPLETANDO EL PAQUETE)
plt.figure(figsize=(10, 6))
plt.scatter(test_df['Distance'], test_df[TARGET], c='gray', alpha=0.5, label='Real')
plt.scatter(test_df['Distance'], test_df['Pred_KNN'], c=COLOR_KNN, marker='x', alpha=0.9, label='KNN')
xr = np.linspace(test_df['Distance'].min(), test_df['Distance'].max(), 100)
plt.plot(xr, 32.44 + 20*np.log10(f) + 20*np.log10(xr/1000), 'k--', label='FSPL')
plt.title('KNN vs Real (Global Downlink)', fontweight='bold')
plt.legend(loc='lower right') # <-- LEYENDA ASEGURADA
plt.grid(True, linestyle='--', alpha=0.5)
plt.savefig(os.path.join(DIR_SALIDA, '6_Scatter_Global.png'), dpi=300); plt.close()

# ==============================================================================
# 7. ANÁLISIS POR CONFIGURACIÓN (LOOP)
# ==============================================================================
print("📂 Generando Análisis por Configuración...")
DIR_CFG = os.path.join(DIR_SALIDA, 'Por_Configuracion')
if not os.path.exists(DIR_CFG): os.makedirs(DIR_CFG)

config_summary = []

for (sf, pwr), grp in test_df.groupby([col_sf, col_pwr]):
    if len(grp) < 10: continue

    nm = f"SF{sf}_P{pwr}"
    m_loc = {}
    for n, c in zip(nombres, cols_res):
        m_loc[n] = calc_metrics(grp[TARGET], grp[c])

    config_summary.append([nm, m_loc['KNN']['RMSE'], m_loc['KNN']['MSE'], m_loc['KNN']['R2'], len(grp)])

    # 1. Tabla KNN Local
    d_knn_loc = [[k, f"{v:.4f}"] for k, v in m_loc['KNN'].items()]
    save_table(d_knn_loc, ['Métrica', 'Valor'], f'Métricas KNN - {nm}', os.path.join(DIR_CFG, f'Tabla_SoloKNN_{nm}.png'))

    # 2. Barras KNN Local
    plt.figure(figsize=(6, 4))
    vals = list(m_loc['KNN'].values())
    keys = list(m_loc['KNN'].keys())
    bars = plt.bar(keys, vals, color=COLOR_KNN, edgecolor='black')
    plt.bar_label(bars, fmt='%.2f', fontweight='bold')
    plt.title(f'Desempeño KNN Downlink - {nm}', fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(DIR_CFG, f'Barras_SoloKNN_{nm}.png'), dpi=300); plt.close()

    # 3. Barras Comparativas Local (RMSE)
    df_loc = pd.DataFrame(m_loc).T
    plt.figure(figsize=(10, 5))
    df_s = df_loc.sort_values('RMSE')
    colors = [COLOR_KNN if x == 'KNN' else '#95a5a6' for x in df_s.index]
    bars = plt.bar(df_s.index, df_s['RMSE'], color=colors, edgecolor='black')
    plt.bar_label(bars, fmt='%.2f', fontweight='bold')
    plt.title(f'Comparativa RMSE - {nm}', fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(os.path.join(DIR_CFG, f'Barras_Comp_RMSE_{nm}.png'), dpi=300); plt.close()

    # 4. Scatter Local
    plt.figure(figsize=(10, 6))
    plt.scatter(grp['Distance'], grp[TARGET], c='gray', alpha=0.5, label='Real')
    plt.scatter(grp['Distance'], grp['Pred_KNN'], c=COLOR_KNN, marker='x', alpha=0.9, label='Pred. KNN')

    # Línea FSPL referencia
    xr = np.linspace(grp['Distance'].min(), grp['Distance'].max(), 100)
    plt.plot(xr, 32.44+20*np.log10(f)+20*np.log10(xr/1000), 'k--', alpha=0.5, label='FSPL')

    plt.title(f'Predicción vs Real Downlink - {nm}', fontweight='bold')
    plt.xlabel('Distancia (m)'); plt.ylabel('Path Loss (dB)')
    plt.legend(loc='lower right') # <-- LEYENDA ASEGURADA
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.savefig(os.path.join(DIR_CFG, f"Scatter_{nm}.png"), dpi=300, bbox_inches='tight'); plt.close()

    # 5. Boxplot Local
    plt.figure(figsize=(12, 6))
    bins = range(int(grp['Distance'].min()), int(grp['Distance'].max()) + BIN_SIZE + 1, BIN_SIZE)
    grp_copy = grp.copy()
    grp_copy['Bin'] = pd.cut(grp_copy['Distance'], bins=bins)

    if not grp_copy['Bin'].isnull().all():
        sns.boxplot(x='Bin', y='Pred_KNN', data=grp_copy, color='#af7ac5') # Morado clarito
        plt.title(f'Distribución KNN por Distancia - {nm}', fontweight='bold')
        plt.xticks(rotation=45)
        plt.grid(axis='y', linestyle='--', alpha=0.5)
        plt.tight_layout()
        plt.savefig(os.path.join(DIR_CFG, f"Boxplot_{nm}.png"), dpi=300); plt.close()

# Guardar resumen CSV
pd.DataFrame(config_summary, columns=['Config', 'RMSE', 'MSE', 'R2', 'Muestras']).to_csv(os.path.join(DIR_SALIDA, 'Resumen_Configs.csv'), index=False)

print(f"\n✅ TEST KNN DOWNLINK GENERAL COMPLETADO.")
print(f"   📂 Resultados en: {DIR_SALIDA}")